# Demo: facturas (PDF / fotos) → Excel con el formato de "Comprobantes de Compras" (ARCA), sin duplicadas
Usa **MarkItDown** para leer los PDFs digitales y una IA (Gemini o Claude) para extraer los campos.
Pensado para probar con pocas facturas (por defecto, 10).

**Antes de correr:**
1. Subí tus facturas a una carpeta de Google Drive (ej: `Mi unidad/facturas_demo`).
2. En Colab, panel izquierdo → 🔑 *Secretos*: agregá `GEMINI_API_KEY` (gratis en aistudio.google.com) o `ANTHROPIC_API_KEY`, con *Acceso desde el notebook* activado.
3. Ejecutá las celdas en orden.

In [ ]:
# 1) Instalar dependencias
!pip install -q "markitdown[pdf]" google-genai anthropic

In [ ]:
# 2) CONFIGURACIÓN: editá solo esta celda
PROVEEDOR = "gemini"                      # "gemini" o "claude"
MODELO = "gemini-3.1-flash-lite"         # claude: "claude-haiku-5-5". Gemini: listá los modelos de tu cuenta si da error 404
CARPETA = "/content/drive/MyDrive/facturas_demo"
SALIDA = "/content/drive/MyDrive/comprobantes_compras.xlsx"
CUIT_PROPIO = None                        # tu CUIT (solo números). Si es None, se toma el receptor más frecuente
MAX_ARCHIVOS = 10                         # poné None para procesar todo
PAUSA_SEG = 1                             # pausa entre archivos (ayuda con los límites del plan gratuito)

import os
from google.colab import drive, userdata
drive.mount("/content/drive")
clave = "GEMINI_API_KEY" if PROVEEDOR == "gemini" else "ANTHROPIC_API_KEY"
os.environ[clave] = userdata.get(clave)
print("Listo. Proveedor:", PROVEEDOR, "| Modelo:", MODELO)

In [ ]:
# 3) Funciones
import base64, hashlib, io, json, re, time
from pathlib import Path
import pandas as pd
from markitdown import MarkItDown

EXT_IMG = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".webp": "image/webp"}
MIN_CHARS = 100

PROMPT = """Sos un extractor de datos de facturas argentinas. Devolvé SOLO un JSON válido
(sin texto extra ni backticks) con estas claves exactas:
{
 "tipo_comprobante": "ej: Factura A, Factura B, Factura C, Nota de Crédito A",
 "punto_venta": entero o null,
 "numero": entero o null,
 "fecha": "YYYY-MM-DD" o null,
 "cuit_emisor": "11 dígitos sin guiones" o null,
 "razon_social_emisor": texto o null,
 "cuit_receptor": "11 dígitos" o null,
 "neto_gravado": número o null,
 "no_gravado": número (importe no gravado) o null,
 "exento": número (operaciones exentas) o null,
 "iva": número total de IVA o null,
 "otros_tributos": número o null,
 "total": número o null,
 "moneda": "ARS", "USD", etc.,
 "cae": texto o null
}
Números con punto decimal y sin separador de miles. Si un dato no está o no se ve, null. No inventes."""

md_conv = MarkItDown()

if PROVEEDOR == "gemini":
    from google import genai
    from google.genai import types
    cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
else:
    from anthropic import Anthropic
    cliente = Anthropic(max_retries=5)


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def num(v):
    if v in (None, ""):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).replace("$", "").replace(" ", "")
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def entero(v):
    n = num(v)
    return int(n) if n is not None else None


def digitos(v):
    d = re.sub(r"\D", "", str(v)) if v is not None else ""
    return d or None


def tipo_norm(t):
    if not t:
        return None
    t = str(t).lower()
    clase = "NC" if "cred" in t or "créd" in t else "ND" if "deb" in t or "déb" in t else "F"
    m = re.search(r"\b([abcemx])\b", t)
    return f"{clase}-{m.group(1).upper() if m else '?'}"


CODIGO_DE_TIPO = {"F-A": 1, "ND-A": 2, "NC-A": 3, "F-B": 6, "ND-B": 7, "NC-B": 8,
                  "F-C": 11, "ND-C": 12, "NC-C": 13, "F-M": 51, "ND-M": 52, "NC-M": 53}
TIPO_DE_CODIGO = {v: k for k, v in CODIGO_DE_TIPO.items()}
NOMBRE_CLASE = {"F": "Factura", "NC": "Nota de Crédito", "ND": "Nota de Débito"}


def etiqueta_tipo(r):
    t = r["tipo"]
    if t in CODIGO_DE_TIPO:
        clase, letra = t.split("-")
        return f"{CODIGO_DE_TIPO[t]} - {NOMBRE_CLASE[clase]} {letra}"
    return r.get("tipo_comprobante") or ""


def fecha_ar(f):
    from datetime import datetime
    try:
        return datetime.strptime(str(f)[:10], "%Y-%m-%d").strftime("%d/%m/%Y")
    except Exception:
        return f


def preparar_imagen(p):
    data, mime = p.read_bytes(), EXT_IMG[p.suffix.lower()]
    if len(data) <= 4_500_000:
        return data, mime
    from PIL import Image
    img, esc = Image.open(io.BytesIO(data)).convert("RGB"), 1.0
    while True:
        buf = io.BytesIO()
        img.resize((int(img.width * esc), int(img.height * esc))).save(buf, "JPEG", quality=85)
        if buf.tell() <= 4_500_000 or esc < 0.3:
            return buf.getvalue(), "image/jpeg"
        esc *= 0.8


def llamar_ia(texto=None, data=None, mime=None):
    """Una sola llamada; reintenta si hay límite de velocidad."""
    for intento in range(4):
        try:
            if PROVEEDOR == "gemini":
                partes = [PROMPT]
                if texto:
                    partes.insert(0, "Texto de la factura:\n\n" + texto[:30000])
                else:
                    partes.insert(0, types.Part.from_bytes(data=data, mime_type=mime))
                r = cliente.models.generate_content(
                    model=MODELO, contents=partes,
                    config=types.GenerateContentConfig(response_mime_type="application/json"))
                salida = r.text
            else:
                if texto:
                    cont = [{"type": "text", "text": "Texto de la factura:\n\n" + texto[:30000]}]
                else:
                    tipo = "document" if mime == "application/pdf" else "image"
                    cont = [{"type": tipo, "source": {"type": "base64", "media_type": mime,
                             "data": base64.standard_b64encode(data).decode()}}]
                cont.append({"type": "text", "text": PROMPT})
                r = cliente.messages.create(model=MODELO, max_tokens=1000,
                                            messages=[{"role": "user", "content": cont}])
                salida = "".join(b.text for b in r.content if b.type == "text")
            salida = re.sub(r"^```(?:json)?|```$", "", salida.strip(), flags=re.M).strip()
            return json.loads(salida)
        except Exception as e:
            if intento < 3 and any(x in str(e).lower() for x in ("429", "rate", "quota", "overloaded", "503")):
                time.sleep(10 * (intento + 1))
                continue
            raise


def procesar(p):
    if p.suffix.lower() == ".pdf":
        try:
            texto = md_conv.convert(str(p)).text_content or ""
        except Exception:
            texto = ""
        if len(texto.strip()) >= MIN_CHARS:
            d, via = llamar_ia(texto=texto), "PDF digital (MarkItDown)"
        else:
            d, via = llamar_ia(data=p.read_bytes(), mime="application/pdf"), "PDF escaneado (visión)"
    else:
        data, mime = preparar_imagen(p)
        d, via = llamar_ia(data=data, mime=mime), "Imagen (visión)"

    r = {
        "archivo": p.name, "lectura": via,
        "tipo": tipo_norm(d.get("tipo_comprobante")),
        "tipo_comprobante": d.get("tipo_comprobante"),
        "punto_venta": entero(d.get("punto_venta")), "numero": entero(d.get("numero")),
        "fecha": d.get("fecha"), "cuit_emisor": digitos(d.get("cuit_emisor")),
        "razon_social_emisor": d.get("razon_social_emisor"),
        "cuit_receptor": digitos(d.get("cuit_receptor")),
        "neto_gravado": num(d.get("neto_gravado")), "no_gravado": num(d.get("no_gravado")),
        "exento": num(d.get("exento")), "iva": num(d.get("iva")),
        "otros_tributos": num(d.get("otros_tributos")), "total": num(d.get("total")),
        "moneda": d.get("moneda") or "ARS", "cae": d.get("cae"),
    }
    obs = []
    # Si el archivo se llama CUIT_TIPO_PV_NUMERO (formato ARCA), esos datos mandan sobre lo leído
    m = re.match(r"^(\d{11})_(\d{1,3})_(\d{1,5})_(\d{1,8})", p.stem)
    if m:
        cuit_f, cod_f, pv_f, num_f = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))
        for campo, val in (("cuit_emisor", cuit_f), ("punto_venta", pv_f), ("numero", num_f)):
            if r[campo] not in (None, "") and r[campo] != val:
                obs.append(f"{campo}: archivo={val} vs leído={r[campo]}")
            r[campo] = val
        if cod_f in TIPO_DE_CODIGO:
            if r["tipo"] and r["tipo"] != TIPO_DE_CODIGO[cod_f]:
                obs.append(f"tipo: archivo={TIPO_DE_CODIGO[cod_f]} vs leído={r['tipo']}")
            r["tipo"] = TIPO_DE_CODIGO[cod_f]
    # Comprobantes C (monotributo): no discriminan IVA, el modelo los lleva con todo en 0 y solo el total
    if r["tipo"] and r["tipo"].endswith("-C"):
        r["neto_gravado"] = r["no_gravado"] = r["exento"] = r["iva"] = 0.0
    if r["tipo"] not in CODIGO_DE_TIPO:
        obs.append("tipo de comprobante no reconocido")
    obs += [f"falta {c}" for c in ("cuit_emisor", "punto_venta", "numero", "fecha", "total")
            if r[c] in (None, "")]
    if r["cuit_emisor"] and len(r["cuit_emisor"]) != 11:
        obs.append("CUIT emisor inválido")
    partes = [r["neto_gravado"], r["no_gravado"], r["exento"], r["iva"], r["otros_tributos"]]
    if r["total"] is not None and any(x is not None for x in partes) and not (r["tipo"] or "").endswith("-C"):
        s = sum(x or 0 for x in partes)
        if abs(s - r["total"]) > 1:
            obs.append(f"suma de componentes ({s:.2f}) ≠ total ({r['total']:.2f})")
    ok = all(r[c] not in (None, "") for c in ("cuit_emisor", "tipo", "punto_venta", "numero")) and "?" not in (r["tipo"] or "?")
    r["clave"] = f"{r['cuit_emisor']}-{r['tipo']}-{r['punto_venta']:05d}-{r['numero']:08d}" if ok else None
    r["observaciones"] = " | ".join(obs)
    return r

print("Funciones cargadas")

In [ ]:
# 4) Procesar la carpeta
archivos = sorted(p for p in Path(CARPETA).rglob("*")
                  if p.is_file() and (p.suffix.lower() == ".pdf" or p.suffix.lower() in EXT_IMG))
print(f"{len(archivos)} archivos encontrados")
if MAX_ARCHIVOS:
    archivos = archivos[:MAX_ARCHIVOS]
    print(f"Demo: se procesan los primeros {len(archivos)}")

# Paso A: duplicados exactos (mismo archivo) por hash
vistos_hash, unicos, duplicadas = {}, [], []
for p in archivos:
    h = sha256(p)
    if h in vistos_hash:
        duplicadas.append({"archivo": p.name, "duplicado_de": vistos_hash[h], "motivo": "archivo idéntico"})
    else:
        vistos_hash[h] = p.name
        unicos.append(p)

# Paso B: extracción
registros, errores = [], []
for i, p in enumerate(unicos, 1):
    try:
        registros.append(procesar(p))
        print(f"[{i}/{len(unicos)}] OK    {p.name}")
    except Exception as e:
        errores.append({"archivo": p.name, "error": str(e)[:300]})
        print(f"[{i}/{len(unicos)}] ERROR {p.name}: {str(e)[:120]}")
    time.sleep(PAUSA_SEG)

# Paso C: duplicados por comprobante (misma factura en archivos distintos)
registros.sort(key=lambda r: (r["observaciones"] != "", r["archivo"]))
vistos, finales = {}, []
for r in registros:
    k = r["clave"]
    if k and k in vistos:
        duplicadas.append({"archivo": r["archivo"], "duplicado_de": vistos[k], "motivo": f"mismo comprobante ({k})"})
    else:
        if k:
            vistos[k] = r["archivo"]
        finales.append(r)

# Paso D: armar el Excel con el formato del modelo "Comprobantes de Compras"
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

recep = [r["cuit_receptor"] for r in finales if r["cuit_receptor"]]
cuit_propio = CUIT_PROPIO or (pd.Series(recep).mode().iloc[0] if recep else "")

finales.sort(key=lambda r: (str(r["fecha"] or "9999"), CODIGO_DE_TIPO.get(r["tipo"], 99),
                            r["punto_venta"] or 0, r["numero"] or 0))

COLS = ["Fecha", "Tipo", "Punto de Venta", "Número Desde", "Número Hasta", "Tipo Doc. Vendedor",
        "Nro. Doc. Vendedor", "Denominación Vendedor", "Tipo Cambio", "Moneda", "Neto Gravado",
        "No Gravado", "Exento", "IVA", "Total"]
ANCHOS = [13, 27.3, 18.2, 15.6, 15.6, 23.4, 23.4, 27.3, 14.3, 7.8, 15.6, 13, 14.3, 14.3, 15.6]


def z(v):
    return 0.0 if v is None else v


wb = Workbook()
ws = wb.active
ws.title = "Comprobantes"
ws.append([f"Comprobantes de Compras - CUIT {cuit_propio}"])
ws.merge_cells("A1:O1")
ws.append(COLS)
for c in ws[2]:
    c.font = Font(bold=True)
for r in finales:
    peso = (r["moneda"] or "ARS").upper() in ("ARS", "$", "PESOS")
    ws.append([
        fecha_ar(r["fecha"]), etiqueta_tipo(r), r["punto_venta"], r["numero"], None, "CUIT",
        int(r["cuit_emisor"]) if r["cuit_emisor"] else None, r["razon_social_emisor"],
        1 if peso else None, "$" if peso else r["moneda"],
        z(r["neto_gravado"]), z(r["no_gravado"]), z(r["exento"]), z(r["iva"]), z(r["total"])])
for fila in ws.iter_rows(min_row=3):
    for c in fila:
        if c.column_letter in "CDGI":
            c.number_format = "0"
        elif c.column_letter in "KLMNO":
            c.number_format = "0.00"
for letra, ancho in zip("ABCDEFGHIJKLMNO", ANCHOS):
    ws.column_dimensions[letra].width = ancho

# Hoja de control: lo que el modelo no tiene (otros tributos, CAE) y qué revisar
amarillo = PatternFill("solid", fgColor="FFF2CC")
wc = wb.create_sheet("Control")
wc.append(["Archivo", "Comprobante", "Otros tributos (no están en el modelo)", "CAE", "Cómo se leyó",
           "Revisar", "Observaciones"])
for c in wc[1]:
    c.font = Font(bold=True)
for r in finales:
    wc.append([r["archivo"], r["clave"], z(r["otros_tributos"]), r["cae"], r["lectura"],
               "SÍ" if r["observaciones"] else "", r["observaciones"]])
    if r["observaciones"]:
        for c in wc[wc.max_row]:
            c.fill = amarillo
wd = wb.create_sheet("Duplicadas")
wd.append(["Archivo", "Duplicado de", "Motivo"])
for d in duplicadas:
    wd.append([d["archivo"], d["duplicado_de"], d["motivo"]])
we = wb.create_sheet("Errores")
we.append(["Archivo", "Error"])
for e in errores:
    we.append([e["archivo"], e["error"]])
for hoja in (wc, wd, we):
    hoja.freeze_panes = "A2"
    for col in hoja.columns:
        ancho = max(len(str(c.value)) if c.value is not None else 0 for c in col)
        hoja.column_dimensions[col[0].column_letter].width = min(max(ancho + 2, 10), 45)
wb.save(SALIDA)

n_rev = sum(1 for r in finales if r["observaciones"])
print(f"Comprobantes: {len(finales)} | a revisar: {n_rev} | duplicadas: {len(duplicadas)} | errores: {len(errores)}")
print("Excel guardado en:", SALIDA)
pd.read_excel(SALIDA, sheet_name="Comprobantes", header=1)

In [ ]:
# 5) (Opcional) ver duplicadas y descargar el Excel
display(pd.DataFrame(duplicadas, columns=["archivo", "duplicado_de", "motivo"]))
from google.colab import files
files.download(SALIDA)